In [5]:
from pycaret.datasets import get_data
dataset = get_data('diamond')

,Carat Weight,Cut,Color,Clarity,Polish,Symmetry,Report,Price
0,1.10,Ideal,H,SI1,VG,EX,GIA,5169
1,0.83,Ideal,H,VS1,ID,ID,AGSL,3470
2,0.85,Ideal,H,SI1,EX,EX,GIA,3183
3,0.91,Ideal,E,SI1,VG,VG,GIA,4370
4,0.83,Ideal,G,SI1,EX,EX,GIA,3171


In [6]:
data = dataset.sample(frac=0.9, random_state=786)
data_unseen = dataset.drop(data.index)

data.reset_index(drop=True, inplace=True)
data_unseen.reset_index(drop=True, inplace=True)

print('Data for Modeling: ' + str(data.shape))
print('Unseen Data For Predictions ' + str(data_unseen.shape))

Data for Modeling: (5400, 8)
Unseen Data For Predictions (600, 8)


In [7]:

from pycaret.regression import *

In [8]:
exp_reg102 = setup(data = data, target = 'Price', session_id=123,
                  normalize = True, transformation = True, transform_target = True, 
                  remove_multicollinearity = True, multicollinearity_threshold = 0.95, 
                  bin_numeric_features = ['Carat Weight'],
                  silent = True)

,Description,Value
0,session_id,123
1,Target,Price
2,Original Data,"(5400, 8)"
3,Missing Values,False
4,Numeric Features,1
5,Categorical Features,6
6,Ordinal Features,False
7,High Cardinality Features,False
8,High Cardinality Method,None
9,Transformed Train Set,"(3779, 39)"


In [9]:
best = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,766.0853,3116466.5642,1704.0975,0.9704,0.0799,0.0576,0.0400
rf,Random Forest Regressor,850.2817,3268110.2457,1770.5503,0.9686,0.0904,0.0657,0.3910
huber,Huber Regressor,940.6199,3651906.6533,1891.7126,0.9640,0.0972,0.0708,0.0860
ridge,Ridge Regression,952.2538,3846277.6391,1934.6314,0.9624,0.0971,0.0715,0.0130
br,Bayesian Ridge,956.6502,3999159.6452,1967.8153,0.9608,0.0972,0.0716,0.0160
lr,Linear Regression,960.2937,4046533.3058,1978.6945,0.9604,0.0973,0.0717,0.8320
et,Extra Trees Regressor,964.0365,4407851.6806,2061.4017,0.9569,0.1054,0.0758,0.4470
dt,Decision Tree Regressor,1002.3791,4693069.5462,2138.8882,0.9538,0.1086,0.0780,0.0190
gbr,Gradient Boosting Regressor,1107.4885,5269002.7327,2255.3276,0.9486,0.1100,0.0832,0.1440
par,Passive Aggressive Regressor,1341.4005,7149372.8557,2588.2842,0.9288,0.1282,0.0964,0.0150


In [10]:
best = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,762.0852,2398824.3414,1548.8138,0.9737,0.0805,0.0581
1,891.4400,5694025.6674,2386.2158,0.9564,0.0790,0.0588
2,747.8227,2302441.7449,1517.3799,0.9789,0.0795,0.0581
3,700.3420,1485922.9146,1218.9844,0.9836,0.0733,0.0558
4,640.5664,1238780.4181,1113.0051,0.9845,0.0784,0.0562
5,835.2700,7118965.6263,2668.1390,0.9392,0.0810,0.0557
6,863.4444,3512642.3609,1874.2045,0.9670,0.0848,0.0621
7,714.5869,2576734.1429,1605.2209,0.9674,0.0752,0.0537
8,740.6562,2290888.7570,1513.5682,0.9775,0.0781,0.0564
9,764.6396,2545439.6689,1595.4434,0.9758,0.0891,0.0610


In [11]:
tuned_best = tune_model(best)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,1072.0825,5208643.9540,2282.2454,0.9428,0.1056,0.0783
1,1168.0408,10184647.4786,3191.3394,0.9219,0.1012,0.0746
2,1066.8059,4198593.9104,2049.0471,0.9615,0.1006,0.0768
3,1046.6599,3769539.8422,1941.5303,0.9584,0.1027,0.0766
4,950.6577,3097542.1181,1759.9836,0.9614,0.1055,0.0761
5,1161.4496,9649673.5473,3106.3924,0.9176,0.1055,0.0760
6,1191.7385,8448576.9201,2906.6436,0.9206,0.1095,0.0806
7,1015.7054,3969738.5659,1992.4203,0.9498,0.1031,0.0760
8,1052.3919,4944536.7619,2223.6314,0.9514,0.1003,0.0730
9,1112.3832,6755637.6223,2599.1609,0.9357,0.1112,0.0788


In [12]:
import matplotlib.pyplot as plt
fig = plt.gcf()
fig.set_size_inches(5, 3.5, forward=True)

plot_model(tuned_best, plot= 'cooks', save=True )

'Cooks Distance.png'

In [13]:
import matplotlib.pyplot as plt
fig = plt.gcf()
fig.set_size_inches(4, 3.5, forward=True)

plot_model(tuned_best, save=True, plot = 'residuals' )

'Residuals.png'

In [14]:
import matplotlib.pyplot as plt
fig = plt.gcf()
fig.set_size_inches(5, 3.5, forward=True)

plot_model(tuned_best, plot = 'learning', save=True )

'Learning Curve.png'

In [15]:

final_model = finalize_model(tuned_best)

In [16]:
print(final_model)

PowerTransformedTargetRegressor(bagging_fraction=0.9, bagging_freq=3,
                                boosting_type='gbdt', class_weight=None,
                                colsample_bytree=1.0, feature_fraction=0.5,
                                importance_type='split', learning_rate=0.4,
                                max_depth=-1, min_child_samples=6,
                                min_child_weight=0.001, min_split_gain=0.3,
                                n_estimators=20, n_jobs=-1, num_leaves=150,
                                objective=None,
                                power_transformer_method='bo...
                                                        importance_type='split',
                                                        learning_rate=0.4,
                                                        max_depth=-1,
                                                        min_child_samples=6,
                                                        min_child_weigh

In [22]:
save_model(final_model,'regresion_best_model_ejercicio')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=None,
          steps=[('dtypes',
                  DataTypes_Auto_infer(categorical_features=[],
                                       display_types=False, features_todrop=[],
                                       id_columns=[], ml_usecase='regression',
                                       numerical_features=[], target='Price',
                                       time_features=[])),
                 ('imputer',
                  Simple_Imputer(categorical_strategy='not_available',
                                 fill_value_categorical=None,
                                 fill_value_numerical=None,
                                 numeric_strategy=...
                                                                          learning_rate=0.4,
                                                                          max_depth=-1,
                                                                          min_child_samples=6,
                                     